## Identifying Top Passage-Level Results from HathiTrust-like Solr Search Engine

### Overview¶

This notebook does accomplishes two tasks:
1. I ran into a small problem with Solr. The version I was using wouldn't accept strings as IDs.
   This provides a work-around, reading the SolrIDs back into HathiTrust passage-level Filenames.
2. More importantly, this notebook emulates HathiTrust's "search in a volume" function, identifying
   the top 2 passages for reach retrieved volume. This strategy is meant to be generous to HathiTrust
   for the purposes of evaluation, treating only the top passages as candidates for review.

In [ ]:
%pip install rank_bm25

### 1. Libraries

In [ ]:
import csv
import os
import glob
import re
from rank_bm25 import BM25Okapi

### 2. Configuration

In [ ]:
METADATA_FILE = 'metadata.csv'
PASSAGE_DIR = 'complete_ecco_tcp_passages'

# Solr Search Results (IDs only)
solr_result_ids = [
    638, 628, 637, 1727, 320, 2288, 1268, 1607, 87, 1609, 
    1267, 2142, 1513, 1899, 1585, 1729, 137, 1124, 1589, 1723, 
    169, 919, 1234, 1620, 2383, 1262, 1905, 2127, 2307, 1938, 
    231, 1537, 38, 170, 2265, 1973, 1897, 1726, 2136, 1125, 
    1890, 2090, 1395, 2264, 37, 2313, 1885, 1900, 1265, 63
]

# The initial query, this time for the "Search Within" ranking
query_string = "sublime, sublimity, terror, awe, astonishment, vast, infinite, obscurity, Burke, taste, genius, imagination, ruins, mountains, tempest, thunder, horror, wonder"

### 3. Build the Solr decoder, mapping ID to Filename

In [ ]:
id_to_filename = {}
current_id = 1  # Matches indexer.py counter

with open(METADATA_FILE, 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Get filename from CSV
        fname = row.get('Filename') # Ensure this matches CSV header exactly
        
        # Add .txt if missing
        if fname and not fname.lower().endswith('.txt'):
            fname += '.txt'
            
        id_to_filename[current_id] = fname
        current_id += 1

print(f"Mapped {len(id_to_filename)} volumes.")

### 4. Define the ranking function

In [ ]:
def simple_tokenizer(text):
    return re.findall(r'\w+', text.lower())

def process_volume_by_id(solr_id, query, top_n=2):
    # 1. Lookup Filename
    filename = id_to_filename.get(solr_id)
    
    if not filename:
        print(f"Error: Solr ID {solr_id} not found in CSV map.")
        return []
        
    # 2. Identify Chunks
    # Remove .txt to match passage convention (e.g. K000039.000)
    vol_stem = filename.replace('.txt', '')
    search_pattern = os.path.join(PASSAGE_DIR, f"{vol_stem}.*.txt")
    chunk_paths = glob.glob(search_pattern)
    
    if not chunk_paths:
        # Quietly skip if no chunks found
        return []

    # 3. Load Texts
    tokenized_corpus = []
    chunk_ids = []
    
    for path in chunk_paths:
        try:
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                tokenized_corpus.append(simple_tokenizer(f.read()))
                chunk_ids.append(os.path.basename(path))
        except:
            continue

    # 4. Rank with BM25
    if not tokenized_corpus:
        return []
        
    bm25 = BM25Okapi(tokenized_corpus)
    tokenized_query = simple_tokenizer(query)
    scores = bm25.get_scores(tokenized_query)
    
    # 5. Sort and Return
    ranked = sorted(zip(chunk_ids, scores), key=lambda x: x[1], reverse=True)
    
    # Format output
    results = []
    for chunk_id, score in ranked[:top_n]:
        if score > 0:
            results.append({
                'solr_vol_id': solr_id,
                'original_filename': filename,
                'chunk_id': chunk_id,
                'score': score
            })
            
    return results

### 5. Execute in bulk for all top volumes

In [ ]:
final_results = []

print(f"Ranking chunks for {len(solr_result_ids)} volumes...")

for sid in solr_result_ids:
    # Ensure ID is integer
    results = process_volume_by_id(int(sid), query_string)
    final_results.extend(results)

# Show Output
print(f"\n--- Top Passages ({len(final_results)}) ---")
for r in final_results[:10]:
    print(f"Vol ID: {r['solr_vol_id']} -> Chunk: {r['chunk_id']} (Score: {r['score']:.4f})")

### 6. Output CSV

In [ ]:
import csv

# 1. Name output file
output_filename = "sublime_-_solr_toy_hathi_-_top_passages.csv"

# 2. Define the columns (must match the keys in the dictionary)
columns = ['solr_vol_id', 'original_filename', 'chunk_id', 'score']

# 3. Write to file
try:
    with open(output_filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        
        # Write the header row
        writer.writeheader()
        
        # Write the data rows
        writer.writerows(final_results)
        
    print(f"Success! Exported {len(final_results)} rows to '{output_filename}'")

except IOError as e:
    print(f"Error saving file: {e}")